# [GIS] SK 월별 통합 데이터의 집계구별 집계 — 2026-10-05

- 분석 목적: 격자 단위 SK 데이터를 강남·춘천 집계구별로 재구성
- 데이터: `sk_통신인구_YYYYMM_통합.csv`, 강남·춘천 `bnd_oa_*_2025_2Q.shp`
- 분석 기간: 2025년 7~12월, 경계 기준일 2025-06-30
- 주요 파라미터: EPSG:5179, 유일한 내부/경계 후보만 배정, 지표별 결측 제외
- 난수 시드: 사용하지 않음(결정적 집계)
- 실행 환경: Python 3.11 이상, `uv sync --extra dev --extra gis`

2025년 7~12월의 기존 outer join 통합 CSV를 보존하고, 2025년 2분기 집계구 경계로 집계합니다.
월마다 **집계구 한 개당 한 행**인 CSV를 각각 만듭니다. 성·연령 12개, 시간대 24개, 요일 7개 지표를 각각 집계합니다.
7~12월의 결과를 하나로 합치는 단계는 지금 실행하지 않습니다.

- `NULL`을 0으로 채우거나 결측이 있는 행을 통째로 삭제하지 않습니다.
- 지표별로 **관측 합계, 유효 격자 평균, 유효 격자 수, 결측률**을 계산합니다.
- 특정 지표가 전부 결측이면 합계·평균을 `NULL`로 남깁니다. 실제 0은 포함합니다.
- 결측률의 분모는 해당 집계구·월에 매칭된 **통합 CSV의 격자 수**입니다.
  SHP 전체 면적에 대한 관측률이 아니며, 세 원자료 모두에 없는 격자는 알 수 없습니다.
- 격자 좌표를 대표점으로 사용합니다. 50m 사각형의 면적 비례 배분은 하지 않습니다.
- 경계 밖/다중 후보 점은 별도로 보존하며 인근 집계구로 강제 배정하지 않습니다.
- 모든 집계구를 매월 남깁니다. 관측 격자가 0개이면 결측률도 정의되지 않아 `NULL`입니다.

관측 합계는 중복 제거된 실인원 수가 아닙니다. 격자당 평균도 거주민의 외출률이나
집계구 전체 면적당 밀도를 뜻하지 않습니다. 월·연령·시간대·요일을 서로 더해 하나의 총인구로 만들지 않습니다.
서로 다른 지표 묶음의 직접 비교는 공통 격자 기준 검증이 별도로 필요합니다.


## 1. 실행 환경

저장소의 `pyproject.toml`과 `uv.lock`에 정의된 환경을 사용합니다.
저장소 루트에서 다음 명령을 실행하고 해당 환경의 Python 커널을 선택하세요.

```bash
uv sync --extra dev --extra gis
```

`pandas`, `numpy`는 기본 의존성이고 `geopandas`, `shapely`는 GIS 추가 의존성입니다.
`pyproj`, `pyogrio`는 GeoPandas 의존성으로 잠금 파일에 포함되어 있습니다.

아래 셀부터 순서대로 실행합니다. 원본 CSV·SHP는 읽기만 하고, 결과는 `sk_통합_집계구` 폴더에 저장합니다.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyproj

print(
    {
        "pandas": pd.__version__,
        "geopandas": gpd.__version__,
        "shapely": shapely.__version__,
        "pyproj": pyproj.__version__,
    }
)

{'pandas': '3.0.2', 'geopandas': '1.1.3', 'shapely': '2.1.2', 'pyproj': '3.7.2'}


## 2. 경로와 분석 기간

저장소의 `data/raw`(통합 CSV)와 `data/raw/reference_data/{강남,춘천}_집계구경계`(SHP)를 기본 입력 경로로 사용합니다.
`OUTPUT_DIR`는 저장소의 `data/processed/sk_통합_집계구` 폴더입니다.
저장소 구조가 다른 환경에서는 `INPUT_DIR`·`BOUNDARY_FILES`만 바꾸세요.
같은 위치에서 재실행하면 이 노트북이 생성하는 동일 이름의 결과 파일을 갱신합니다.


In [2]:
def existing_path(path):
    """macOS 한글 파일명의 NFC/NFD 차이를 처리합니다."""
    path = Path(path).expanduser().absolute()
    if path.exists():
        return path
    current = Path(path.anchor)
    for part in path.parts[1:]:
        direct = current / part
        if direct.exists():
            current = direct
            continue
        matches = [
            p
            for p in current.iterdir()
            if unicodedata.normalize("NFC", p.name) == unicodedata.normalize("NFC", part)
        ]
        if len(matches) != 1:
            raise FileNotFoundError(f"경로를 확인하세요: {path}")
        current = matches[0]
    return current


# 저장소의 data/processed 아래에 저장해 원자료·결과가 Git에 추가되지 않도록 합니다.
REPO_ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "pyproject.toml").is_file() and (p / "src").is_dir()
    ),
    Path.cwd(),
)
INPUT_DIR = existing_path(REPO_ROOT / "data" / "raw")
BOUNDARY_FILES = {
    "강남구": existing_path(
        REPO_ROOT / "data/raw/reference_data/강남_집계구경계/bnd_oa_11230_2025_2Q.shp"
    ),
    "춘천시": existing_path(
        REPO_ROOT / "data/raw/reference_data/춘천_집계구경계/bnd_oa_32010_2025_2Q.shp"
    ),
}
MONTHS = [f"2025{month:02d}" for month in range(7, 13)]
SK_CRS = "EPSG:5179"  # 이번 SK 자료에서 확인한 UTM-K, 단위 m
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "sk_통합_집계구"
QC_DIR = OUTPUT_DIR / "검증자료"
INPUT_FILES = {ym: existing_path(INPUT_DIR / f"sk_통신인구_{ym}_통합.csv") for ym in MONTHS}

KEYS = ["STD_YM", "BLOCK_CD", "X_COORD", "Y_COORD"]
XY = ["X_COORD", "Y_COORD"]
AGE_COLS = [
    f"{sex}_FLOW_POP_CNT_{age}"
    for sex in ["MAN", "WMAN"]
    for age in ["10G", "20G", "30G", "40G", "50G", "60GU"]
]
HOUR_COLS = [f"TMST_{hour:02d}" for hour in range(24)]
WEEKDAY_COLS = [f"FLOW_POP_CNT_{day}" for day in ["MON", "TUS", "WED", "THU", "FRI", "SAT", "SUN"]]
GROUPS = {"age": AGE_COLS, "hour": HOUR_COLS, "weekday": WEEKDAY_COLS}
VALUE_COLS = AGE_COLS + HOUR_COLS + WEEKDAY_COLS
print("입력:", INPUT_DIR)
print("결과:", OUTPUT_DIR)

## 3. 집계구 경계 점검

집계구 식별자는 `TOT_OA_CD`를 사용합니다. SK의 `BLOCK_CD`를 잘라서 연결하지 않습니다.
잘못된 도형이나 중복 집계구 코드가 발견되면 자동 수정하지 않고 중단합니다.
면적은 EPSG:5179에서 계산하며, 행정 경계 안의 비주거 지역도 포함합니다.


In [3]:
boundary_parts = []
for region, path in BOUNDARY_FILES.items():
    for suffix in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        if not path.with_suffix(suffix).exists():
            raise FileNotFoundError(path.with_suffix(suffix))
    frame = gpd.read_file(path)
    required = ["BASE_DATE", "ADM_CD", "TOT_OA_CD"]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"{region}: 집계구 코드 또는 기준일 열 누락")
    if frame.crs is None:
        raise ValueError(f"{region}: 좌표계가 없습니다.")
    frame = frame[required + ["geometry"]].to_crs(SK_CRS)
    if frame[required].isna().any().any():
        raise ValueError(f"{region}: 경계 속성 결측")
    for col in required:
        frame[col] = frame[col].astype("string").str.strip()
        if frame[col].eq("").any():
            raise ValueError(f"{region}: 빈 {col}")
    if (frame.geometry.isna() | frame.geometry.is_empty | ~frame.geometry.is_valid).any():
        raise ValueError(f"{region}: 비어 있거나 유효하지 않은 경계를 먼저 확인하세요.")
    if not frame.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all():
        raise ValueError(f"{region}: 폴리곤이 아닌 도형")
    frame["REGION"] = region
    boundary_parts.append(frame)

boundaries = gpd.GeoDataFrame(pd.concat(boundary_parts, ignore_index=True), crs=SK_CRS)
if boundaries["TOT_OA_CD"].duplicated().any():
    raise ValueError("집계구 코드 중복: 경계 파일을 확인하세요.")
boundaries["AREA_M2"] = boundaries.geometry.area
if boundaries["AREA_M2"].le(0).any():
    raise ValueError("면적이 0 이하인 집계구")
oa_meta = boundaries[["REGION", "BASE_DATE", "ADM_CD", "TOT_OA_CD", "AREA_M2"]].copy()
print(oa_meta.groupby(["REGION", "BASE_DATE"]).size().rename("집계구수").to_string())

REGION  BASE_DATE
강남구     20250630     1098
춘천시     20250630      600


## 4. 월별 CSV 읽기와 검증

긴 코드와 기준년월은 문자열로 유지합니다. 좌표는 숫자로 읽되 반올림하지 않습니다.
동일 월에 동일 좌표가 여러 행이면 합계가 중복될 수 있으므로 중단합니다.
현재 파일은 묶음별로 전부 존재하거나 전부 결측인 패턴이지만, 향후 부분 결측이 생겨도
각 지표의 유효 값으로 집계하고 부분 결측 격자 수를 별도로 기록합니다.


In [4]:
def read_month(ym, path):
    frame = pd.read_csv(
        path,
        encoding="utf-8-sig",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
        na_values=["NULL", "null"],
        keep_default_na=True,
    )
    if set(frame.columns) != set(KEYS + VALUE_COLS):
        raise ValueError(
            f"{path.name}: 예상 열과 다릅니다. 누락={set(KEYS + VALUE_COLS) - set(frame.columns)}, 추가={set(frame.columns) - set(KEYS + VALUE_COLS)}"
        )
    if frame.empty or frame[KEYS].isna().any().any():
        raise ValueError(f"{path.name}: 빈 파일 또는 결합 키 결측")
    if not frame["STD_YM"].eq(ym).all():
        raise ValueError(f"{path.name}: 기준년월 불일치")
    if frame["BLOCK_CD"].str.strip().eq("").any():
        raise ValueError(f"{path.name}: 빈 소지역 코드")
    for col in XY + VALUE_COLS:
        frame[col] = pd.to_numeric(frame[col], errors="raise")
    if not np.isfinite(frame[XY].to_numpy(dtype=float)).all():
        raise ValueError(f"{path.name}: 유효하지 않은 좌표")
    values = frame[VALUE_COLS].to_numpy(dtype=float)
    if np.isinf(values).any() or np.any(values < 0):
        raise ValueError(f"{path.name}: 무한대 또는 음수 지표")
    if frame.duplicated(KEYS).any() or frame.duplicated(XY).any():
        raise ValueError(f"{path.name}: 중복 키 또는 중복 좌표")
    return frame


# 한 번에 여러 달의 전체 지표를 메모리에 두지 않고 좌표 합집합만 유지합니다.
coordinate_parts, input_checks = [], []
for ym, path in INPUT_FILES.items():
    frame = read_month(ym, path)
    coordinate_parts.append(frame[XY])
    record = {"STD_YM": ym, "N_INPUT": len(frame)}
    for group, cols in GROUPS.items():
        n_present = frame[cols].notna().sum(axis=1)
        record[f"{group}_all_missing"] = int(n_present.eq(0).sum())
        record[f"{group}_partial"] = int(n_present.between(1, len(cols) - 1).sum())
    input_checks.append(record)
    del frame
coords = pd.concat(coordinate_parts, ignore_index=True).drop_duplicates().reset_index(drop=True)
coords.index.name = "POINT_ID"
del coordinate_parts
input_checks = pd.DataFrame(input_checks)
print(input_checks.to_string(index=False))
print("6개월 좌표 합집합:", f"{len(coords):,}")

STD_YM  N_INPUT  age_all_missing  age_partial  hour_all_missing  hour_partial  weekday_all_missing  weekday_partial
202507   117825            19696            0             23726             0                    0                0
202508   118958            19843            0             23584             0                    0                0
202509   119865            20396            0             24781             0                    0                0
202510   121155            20405            0             23274             0                  512                0
202511   122248            21804            0             26426             0                    0                0
202512   121267            22922            0             28722             0                    0                0
6개월 좌표 합집합: 126,405


## 5. 좌표 → 집계구 공간 매칭

1. 폴리곤 내부(`within`)에 들어가는 집계구가 하나면 배정합니다.
2. 내부 후보가 없으면 경계 포함(`intersects`) 조건으로 재확인합니다.
3. 경계 포함 후보가 하나면 `boundary_unique`로 배정합니다.
4. 내부/경계 후보가 여러 개면 모호한 점으로 제외하고 후보 목록을 보존합니다.
5. 어느 집계구에도 닿지 않는 점은 `outside`로 보존합니다.

한 점을 여러 집계구에 중복 합산하지 않습니다. 모든 달의 좌표 합집합을 사용하므로
7월에 없고 다른 월에만 등장하는 격자도 매칭됩니다.


In [5]:
points = gpd.GeoDataFrame(
    coords.copy(), geometry=gpd.points_from_xy(coords.X_COORD, coords.Y_COORD), crs=SK_CRS
)
right = boundaries[["TOT_OA_CD", "geometry"]]
point_map = coords.copy()
point_map["TOT_OA_CD"] = pd.Series(pd.NA, index=point_map.index, dtype="string")
point_map["MATCH_STATUS"] = "outside"
point_map["N_CANDIDATES"] = 0

inside = gpd.sjoin(points, right, how="inner", predicate="within")
inside_counts = inside.groupby(level=0).size()
point_map.loc[inside_counts.index, "N_CANDIDATES"] = inside_counts
single_inside = inside.loc[inside.index.isin(inside_counts[inside_counts.eq(1)].index)]
point_map.loc[single_inside.index, "TOT_OA_CD"] = single_inside["TOT_OA_CD"]
point_map.loc[single_inside.index, "MATCH_STATUS"] = "within"
point_map.loc[inside_counts[inside_counts.gt(1)].index, "MATCH_STATUS"] = "ambiguous_interior"

remaining = points.loc[~points.index.isin(inside_counts.index)]
edge = gpd.sjoin(remaining, right, how="inner", predicate="intersects")
edge_counts = edge.groupby(level=0).size()
point_map.loc[edge_counts.index, "N_CANDIDATES"] = edge_counts
single_edge = edge.loc[edge.index.isin(edge_counts[edge_counts.eq(1)].index)]
point_map.loc[single_edge.index, "TOT_OA_CD"] = single_edge["TOT_OA_CD"]
point_map.loc[single_edge.index, "MATCH_STATUS"] = "boundary_unique"
point_map.loc[edge_counts[edge_counts.gt(1)].index, "MATCH_STATUS"] = "ambiguous_boundary"

candidates = (
    pd.concat(
        [
            inside[["TOT_OA_CD"]].assign(PREDICATE="within"),
            edge[["TOT_OA_CD"]].assign(PREDICATE="intersects"),
        ]
    )
    .rename_axis("POINT_ID")
    .reset_index()
)
ambiguous_ids = point_map.index[point_map.MATCH_STATUS.str.startswith("ambiguous")]
ambiguous_candidates = candidates[candidates.POINT_ID.isin(ambiguous_ids)].copy()
point_map = point_map.reset_index()
assert not point_map.duplicated(XY).any()
assert point_map.loc[point_map.TOT_OA_CD.notna(), "N_CANDIDATES"].eq(1).all()
print(point_map.MATCH_STATUS.value_counts().to_string())
if len(ambiguous_ids):
    warnings.warn(
        f"{len(ambiguous_ids)}개 다중 후보 좌표를 집계에서 제외합니다. 후보 목록을 확인하세요."
    )

MATCH_STATUS
within     126289
outside       116


## 6. 집계 함수: 합계·평균·유효 수·결측률

예: `TMST_00_sum`은 관측된 0시 값의 합, `TMST_00_mean`은 그 합을
`TMST_00_n_valid`로 나눈 값입니다. `TMST_00_missing_rate`는
`(N_GRID - TMST_00_n_valid) / N_GRID`이며 0~1 비율입니다.

묶음별 `*_n_complete`는 그 묶음의 모든 지표가 있는 격자 수입니다.
`N_COMMON_ALL`은 43개 지표가 모두 존재하는 격자 수입니다.
유효 격자가 같다고 가정하지 않고 지표별 분모를 저장합니다.


In [6]:
def aggregate_values(frame, index, value_cols):
    """NaN은 제외하지만 실제 0은 포함. 전부 NaN이면 합계·평균도 NaN."""
    grouped = frame.groupby("TOT_OA_CD", sort=False)[value_cols]
    n_grid = frame.groupby("TOT_OA_CD").size().reindex(index, fill_value=0).astype("int64")
    sums = grouped.sum(min_count=1).reindex(index)
    means = grouped.mean().reindex(index)
    counts = grouped.count().reindex(index, fill_value=0).astype("int64")
    missing_rates = counts.rsub(n_grid, axis=0).div(n_grid.replace(0, np.nan), axis=0)
    stats = pd.concat(
        [
            sums.add_suffix("_sum"),
            means.add_suffix("_mean"),
            counts.add_suffix("_n_valid"),
            missing_rates.add_suffix("_missing_rate"),
        ],
        axis=1,
    )
    order = [
        f"{col}_{suffix}"
        for col in value_cols
        for suffix in ["sum", "mean", "n_valid", "missing_rate"]
    ]
    return n_grid, stats[order]


# 결측/0/미관측 집계구에 대한 작은 검증: 실제 분석에서 중요한 경계 조건입니다.
toy = pd.DataFrame({"TOT_OA_CD": ["A", "A", "A", "B"], "v": [0.0, 10.0, np.nan, np.nan]})
toy_n, toy_stats = aggregate_values(toy, pd.Index(["A", "B", "C"], name="TOT_OA_CD"), ["v"])
assert toy_stats.loc["A", "v_sum"] == 10
assert toy_stats.loc["A", "v_mean"] == 5
assert toy_stats.loc["A", "v_n_valid"] == 2
assert np.isclose(toy_stats.loc["A", "v_missing_rate"], 1 / 3)
assert pd.isna(toy_stats.loc["B", "v_sum"]) and toy_stats.loc["B", "v_missing_rate"] == 1
assert toy_n.loc["C"] == 0 and pd.isna(toy_stats.loc["C", "v_missing_rate"])
print("결측·실제 0·관측 없는 집계구 처리 검증 통과")

결측·실제 0·관측 없는 집계구 처리 검증 통과


## 7. 월별 독립 집계와 총량 검증

매월 모든 집계구를 유지합니다. 집계된 합계·유효 수가 공간 매칭에 성공한
원자료의 합계·유효 수와 일치하는지 43개 지표 모두 확인합니다.
미매칭 점은 집계에 넣지 않고 원래 값과 제외 사유를 월별 별도 CSV로 보존합니다.
결과는 `monthly_results["202507"]`처럼 월별로 보관하며, 여러 달의 행을 합치지 않습니다.


In [7]:
oa_index = pd.Index(oa_meta.TOT_OA_CD, name="TOT_OA_CD")
monthly_results, audit_rows, monthly_excluded = {}, [], {}
for ym, path in INPUT_FILES.items():
    frame = read_month(ym, path)
    joined = frame.merge(point_map, on=XY, how="left", validate="one_to_one")
    assert joined.MATCH_STATUS.notna().all()
    matched = joined[joined.TOT_OA_CD.notna()].copy()
    excluded = joined[joined.TOT_OA_CD.isna()].copy()
    monthly_excluded[ym] = excluded
    n_grid, stats = aggregate_values(matched, oa_index, VALUE_COLS)
    coverage = pd.DataFrame({"N_GRID": n_grid})

    for group, cols in GROUPS.items():
        n_present = matched[cols].notna().sum(axis=1)
        complete = (
            n_present.eq(len(cols))
            .groupby(matched.TOT_OA_CD)
            .sum()
            .reindex(oa_index, fill_value=0)
            .astype("int64")
        )
        partial = (
            n_present.between(1, len(cols) - 1)
            .groupby(matched.TOT_OA_CD)
            .sum()
            .reindex(oa_index, fill_value=0)
            .astype("int64")
        )
        coverage[f"{group}_n_complete"] = complete
        coverage[f"{group}_n_partial"] = partial
        coverage[f"{group}_n_all_missing"] = n_grid - complete - partial
        coverage[f"{group}_incomplete_rate"] = (n_grid - complete).div(n_grid.replace(0, np.nan))
    coverage["N_COMMON_ALL"] = (
        matched[VALUE_COLS]
        .notna()
        .all(axis=1)
        .groupby(matched.TOT_OA_CD)
        .sum()
        .reindex(oa_index, fill_value=0)
        .astype("int64")
    )
    coverage["COMMON_ALL_RATE"] = coverage.N_COMMON_ALL.div(n_grid.replace(0, np.nan))
    coverage["HAS_SK_POINTS"] = n_grid.gt(0)

    result = oa_meta.set_index("TOT_OA_CD").join(coverage).join(stats).reset_index()
    result.insert(0, "STD_YM", ym)
    assert len(result) == len(oa_meta)
    assert not result.duplicated(["STD_YM", "TOT_OA_CD"]).any()
    assert int(n_grid.sum()) == len(matched)
    max_sum_error = 0.0
    for col in VALUE_COLS:
        counts = result[f"{col}_n_valid"]
        sums = result[f"{col}_sum"]
        means = result[f"{col}_mean"]
        assert int(counts.sum()) == int(matched[col].count())
        assert counts.le(result.N_GRID).all()
        assert sums[counts.eq(0)].isna().all() and means[counts.eq(0)].isna().all()
        np.testing.assert_allclose(
            means, sums / counts.replace(0, np.nan), rtol=1e-10, atol=1e-9, equal_nan=True
        )
        source_sum = matched[col].sum(min_count=1)
        output_sum = sums.sum(min_count=1)
        np.testing.assert_allclose(output_sum, source_sum, rtol=1e-10, atol=1e-6, equal_nan=True)
        if pd.notna(source_sum):
            max_sum_error = max(max_sum_error, abs(output_sum - source_sum))
        rates = result[f"{col}_missing_rate"]
        assert rates.dropna().between(0, 1).all()
        assert rates[result.N_GRID.eq(0)].isna().all()
    audit = {
        "STD_YM": ym,
        "N_INPUT": len(frame),
        "N_MATCHED": len(matched),
        "N_EXCLUDED": len(excluded),
        "MATCH_RATE": len(matched) / len(frame),
        "N_OA": len(result),
        "N_OA_NO_POINTS": int(n_grid.eq(0).sum()),
        "MAX_SUM_ABS_ERROR": max_sum_error,
    }
    for status in [
        "within",
        "boundary_unique",
        "outside",
        "ambiguous_interior",
        "ambiguous_boundary",
    ]:
        audit[f"N_{status.upper()}"] = int(joined.MATCH_STATUS.eq(status).sum())
    audit_rows.append(audit)
    monthly_results[ym] = result.sort_values(["REGION", "TOT_OA_CD"]).reset_index(drop=True)
    print(
        f"{ym}: 입력 {len(frame):,}, 매칭 {len(matched):,}, 제외 {len(excluded):,}, 관측 없는 집계구 {int(n_grid.eq(0).sum()):,}"
    )

audit = pd.DataFrame(audit_rows)
assert set(monthly_results) == set(MONTHS)
assert all(len(result) == len(oa_meta) for result in monthly_results.values())
print("\n월별 결과 크기:", {ym: result.shape for ym, result in monthly_results.items()})
print(audit.to_string(index=False))
print("\n모든 월·지표의 합계 및 유효 격자 수 보존 검증 통과")

202507: 입력 117,825, 매칭 117,716, 제외 109, 관측 없는 집계구 3


202508: 입력 118,958, 매칭 118,847, 제외 111, 관측 없는 집계구 3


202509: 입력 119,865, 매칭 119,753, 제외 112, 관측 없는 집계구 3


202510: 입력 121,155, 매칭 121,042, 제외 113, 관측 없는 집계구 3


202511: 입력 122,248, 매칭 122,138, 제외 110, 관측 없는 집계구 3


202512: 입력 121,267, 매칭 121,169, 제외 98, 관측 없는 집계구 3

월별 결과 크기: {'202507': (1698, 194), '202508': (1698, 194), '202509': (1698, 194), '202510': (1698, 194), '202511': (1698, 194), '202512': (1698, 194)}
STD_YM  N_INPUT  N_MATCHED  N_EXCLUDED  MATCH_RATE  N_OA  N_OA_NO_POINTS  MAX_SUM_ABS_ERROR  N_WITHIN  N_BOUNDARY_UNIQUE  N_OUTSIDE  N_AMBIGUOUS_INTERIOR  N_AMBIGUOUS_BOUNDARY
202507   117825     117716         109    0.999075  1698               3       1.862645e-09    117716                  0        109                     0                     0
202508   118958     118847         111    0.999067  1698               3       1.862645e-09    118847                  0        111                     0                     0
202509   119865     119753         112    0.999066  1698               3       2.328306e-10    119753                  0        112                     0                     0
202510   121155     121042         113    0.999067  1698               3       1.862645e-09    

## 8. 월별 CSV 6개 저장

`sk_통합_집계구` 폴더에 아래 6개 파일을 저장합니다. 각 파일은 해당 월의 집계구만 포함합니다.

- `sk_통신인구_202507_집계구.csv`
- `sk_통신인구_202508_집계구.csv`
- `sk_통신인구_202509_집계구.csv`
- `sk_통신인구_202510_집계구.csv`
- `sk_통신인구_202511_집계구.csv`
- `sk_통신인구_202512_집계구.csv`

같은 폴더 아래 `검증자료`에는 매칭 검증표, 원자료 결측 점검표, 좌표 매핑,
월별 미매칭 원본 행, 다중 후보 목록, 열 설명, 실행 정보를 저장합니다.
6개월을 합친 집계 데이터 파일은 생성하지 않습니다.

모든 CSV는 UTF-8 BOM, 결측은 `NULL`로 저장합니다. 코드 열은 Excel에서 직접 열면
숫자로 인식될 수 있으므로 Python에서는 아래 재로딩 코드처럼 문자열 자료형을 지정하세요.
원자료에 없는 성·연령/시간대 값을 다른 묶음에서 추정해 채우지 않습니다.


In [8]:
descriptions = {
    "STD_YM": "기준년월",
    "REGION": "지역명",
    "BASE_DATE": "집계구 경계 기준일",
    "ADM_CD": "경계 파일의 행정동 코드",
    "TOT_OA_CD": "집계구 코드",
    "AREA_M2": "집계구 경계 전체 면적(m²), 비주거 지역 포함",
    "N_GRID": "해당 집계구·월에 매칭된 통합 CSV 격자 수",
    "N_COMMON_ALL": "43개 지표가 모두 유효한 공통 격자 수",
    "COMMON_ALL_RATE": "N_COMMON_ALL / N_GRID. N_GRID=0이면 NULL",
    "HAS_SK_POINTS": "해당 월에 매칭된 SK 격자 존재 여부",
}
labels = {"age": "성·연령", "hour": "시간대", "weekday": "요일"}
for group, cols in GROUPS.items():
    descriptions[f"{group}_n_complete"] = f"{labels[group]} 지표가 전부 유효한 격자 수"
    descriptions[f"{group}_n_partial"] = f"{labels[group]} 일부 지표만 결측인 격자 수"
    descriptions[f"{group}_n_all_missing"] = f"{labels[group]} 지표가 전부 결측인 격자 수"
    descriptions[f"{group}_incomplete_rate"] = (
        f"{labels[group]} 지표 중 하나라도 결측인 격자 비율. 분모 N_GRID, 0개면 NULL"
    )
    for col in cols:
        descriptions[f"{col}_sum"] = f"{col} 관측 합계. 전부 결측이면 NULL"
        descriptions[f"{col}_mean"] = f"{col} 유효 격자 평균. 실제 0 포함, 전부 결측이면 NULL"
        descriptions[f"{col}_n_valid"] = f"{col} 유효한 격자 수. 실제 0 포함"
        descriptions[f"{col}_missing_rate"] = (
            f"(N_GRID - {col}_n_valid) / N_GRID. N_GRID=0이면 NULL"
        )
result_columns = monthly_results[MONTHS[0]].columns
assert all(result.columns.equals(result_columns) for result in monthly_results.values())
dictionary = pd.DataFrame(
    {"COLUMN": result_columns, "DESCRIPTION": [descriptions[c] for c in result_columns]}
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
saved_files = {}
for ym, result in monthly_results.items():
    output_path = OUTPUT_DIR / f"sk_통신인구_{ym}_집계구.csv"
    result.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="NULL")
    saved_files[ym] = output_path

qc_outputs = {
    "SK_집계구_매칭검증.csv": audit,
    "SK_원자료_결측점검.csv": input_checks,
    "SK_좌표_집계구_매핑.csv": point_map,
    "SK_다중후보_집계구.csv": ambiguous_candidates,
    "SK_집계구_열설명.csv": dictionary,
}
for ym, excluded in monthly_excluded.items():
    qc_outputs[f"SK_미매칭_원본행_{ym}.csv"] = excluded
for name, frame in qc_outputs.items():
    frame.to_csv(QC_DIR / name, index=False, encoding="utf-8-sig", na_rep="NULL")

source_paths = list(INPUT_FILES.values()) + [
    p.with_suffix(ext)
    for p in BOUNDARY_FILES.values()
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]
]
manifest = {
    "created_at": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(),
    "months": MONTHS,
    "crs": SK_CRS,
    "versions": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "geopandas": gpd.__version__,
        "shapely": shapely.__version__,
        "pyproj": pyproj.__version__,
    },
    "sources": [
        {"path": str(p), "size_bytes": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
        for p in source_paths
    ],
    "mapping": "within unique, else intersects unique; ambiguous/outside excluded and retained",
    "aggregation": "per-metric sum(min_count=1), mean, count; missing rate uses mapped union grids",
    "no_points": "all census areas retained monthly; counts 0, sum/mean/missing_rate NULL",
    "monthly_shapes": {ym: list(result.shape) for ym, result in monthly_results.items()},
    "monthly_output_files": {ym: str(path) for ym, path in saved_files.items()},
    "months_combined": False,
    "source_files_modified": False,
}
(QC_DIR / "실행정보.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

# 각 파일을 재로딩해서 해당 월만 포함하고 코드·행 수·결측·숫자가 보존되는지 확인합니다.
numeric_cols = [
    c
    for c in result_columns
    if c not in ["STD_YM", "REGION", "BASE_DATE", "ADM_CD", "TOT_OA_CD", "HAS_SK_POINTS"]
]
for ym, output_path in saved_files.items():
    result = monthly_results[ym]
    reloaded = pd.read_csv(
        output_path,
        encoding="utf-8-sig",
        na_values=["NULL"],
        dtype={c: "string" for c in ["STD_YM", "BASE_DATE", "ADM_CD", "TOT_OA_CD", "REGION"]},
    )
    assert reloaded.shape == result.shape
    assert reloaded["STD_YM"].eq(ym).all()
    assert not reloaded["TOT_OA_CD"].duplicated().any()
    assert reloaded.TOT_OA_CD.tolist() == result.TOT_OA_CD.tolist()
    np.testing.assert_allclose(
        reloaded[numeric_cols].to_numpy(dtype=float),
        result[numeric_cols].to_numpy(dtype=float),
        equal_nan=True,
        rtol=1e-10,
        atol=1e-9,
    )
    print(f"{output_path.name}: {len(reloaded):,}행, 재로딩 검증 통과")
print("저장 위치:", OUTPUT_DIR)
print("월별 집계 CSV:", len(saved_files), "개 / 검증·설명 파일:", len(qc_outputs) + 1, "개")

sk_통신인구_202507_집계구.csv: 1,698행, 재로딩 검증 통과
sk_통신인구_202508_집계구.csv: 1,698행, 재로딩 검증 통과
sk_통신인구_202509_집계구.csv: 1,698행, 재로딩 검증 통과
sk_통신인구_202510_집계구.csv: 1,698행, 재로딩 검증 통과
sk_통신인구_202511_집계구.csv: 1,698행, 재로딩 검증 통과
sk_통신인구_202512_집계구.csv: 1,698행, 재로딩 검증 통과
월별 집계 CSV: 6 개 / 검증·설명 파일: 12 개


## 9. 결과 확인 예시

`age_incomplete_rate`는 성·연령 묶음 중 하나라도 값이 없는 격자 비율입니다.
현재 데이터는 묶음 내 부분 결측이 없어, 그 묶음이 통째로 없는 격자 비율과 같습니다.
결측이 많은 집계구를 저활동·고립 위험 지역으로 바로 해석하지 마세요.
관측 격자가 0개인 집계구도 별도로 확인합니다.


In [9]:
show_cols = [
    "STD_YM",
    "REGION",
    "TOT_OA_CD",
    "N_GRID",
    "age_n_complete",
    "hour_n_complete",
    "weekday_n_complete",
    "N_COMMON_ALL",
    "age_incomplete_rate",
    "hour_incomplete_rate",
    "TMST_00_sum",
    "TMST_00_mean",
]
SELECT_MONTH = "202507"  # 확인할 월만 선택
selected = monthly_results[SELECT_MONTH]
print(selected[show_cols].head(10).to_string(index=False))
print("\n선택 월의 관측 격자가 없는 집계구 수:", int(selected.N_GRID.eq(0).sum()))
print("\n선택 월의 지역별 성·연령 결측률 분포 (관측 격자가 있는 집계구만):")
print(
    selected.loc[selected.N_GRID.gt(0)]
    .groupby("REGION")["age_incomplete_rate"]
    .agg(["median", "max"])
    .round(4)
    .to_string()
)

STD_YM REGION      TOT_OA_CD  N_GRID  age_n_complete  hour_n_complete  weekday_n_complete  N_COMMON_ALL  age_incomplete_rate  hour_incomplete_rate  TMST_00_sum  TMST_00_mean
202507    강남구 11230510010001     443             436              436                 443           436             0.015801              0.015801       377.77      0.866445
202507    강남구 11230510010101       7               7                7                   7             7             0.000000              0.000000        22.83      3.261429
202507    강남구 11230510010102       2               2                2                   2             2             0.000000              0.000000         3.99      1.995000
202507    강남구 11230510010103       4               4                4                   4             4             0.000000              0.000000         7.30      1.825000
202507    강남구 11230510010104       3               3                3                   3             3             0.000000      

## 10. 월 전체 통합

In [ ]:
from pathlib import Path
import pandas as pd

# 기존 노트북의 OUTPUT_DIR = ... / "sk_통합_집계구"
folder = Path(OUTPUT_DIR)
months = [f"2025{month:02d}" for month in range(7, 13)]

# 집계구·행정동 코드와 날짜 표기를 문자열로 보존
string_cols = {
    col: "string"
    for col in ["STD_YM", "TOT_OA_CD", "ADM_CD", "BASE_DATE", "REGION"]
}

parts = []
expected_columns = None

for ym in months:
    path = folder / f"sk_통신인구_{ym}_집계구.csv"
    if not path.is_file():
        raise FileNotFoundError(f"월별 파일이 없습니다: {path}")

    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        dtype=string_cols,
        na_values=["NULL"],
    )

    required = {"STD_YM", "TOT_OA_CD", "REGION"}
    if not required.issubset(df.columns):
        raise ValueError(f"{path.name}: 필수 열이 없습니다.")

    if df.empty or df[["STD_YM", "TOT_OA_CD"]].isna().any().any():
        raise ValueError(f"{path.name}: 빈 파일 또는 기준년월·집계구 코드 결측")

    if not df["STD_YM"].eq(ym).all():
        raise ValueError(f"{path.name}: 파일명과 데이터의 기준년월이 다릅니다.")

    if expected_columns is None:
        expected_columns = df.columns.tolist()
    elif set(df.columns) != set(expected_columns):
        raise ValueError(f"{path.name}: 다른 월과 열 구성이 다릅니다.")

    parts.append(df[expected_columns])
    print(f"{ym}: {len(df):,}행 읽기 완료")

# 월별 행을 세로로 연결 — 재합산하거나 평균을 다시 계산하지 않음
combined = pd.concat(parts, ignore_index=True)

if combined.duplicated(["STD_YM", "TOT_OA_CD"]).any():
    raise ValueError("같은 월·집계구 조합이 중복되어 있습니다.")

combined = combined.sort_values(
    ["STD_YM", "REGION", "TOT_OA_CD"]
).reset_index(drop=True)

output_path = folder / "sk_통신인구_전체_집계구.csv"
combined.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    na_rep="NULL",
)

print(f"\n저장 완료: {output_path}")
print(f"전체 {len(combined):,}행 × {len(combined.columns):,}열")
print(combined.groupby("STD_YM").size().rename("집계구 수"))

202507: 1,698행 읽기 완료
202508: 1,698행 읽기 완료
202509: 1,698행 읽기 완료
202510: 1,698행 읽기 완료
202511: 1,698행 읽기 완료
202512: 1,698행 읽기 완료

전체 10,188행 × 194열
STD_YM
202507    1698
202508    1698
202509    1698
202510    1698
202511    1698
202512    1698
Name: 집계구 수, dtype: int64
